# EDA: NYC Yellow Taxi Trips

Covers every month used by the pipeline (2024-01 to 2026-09, ~123M trips). 

**Train** = 2024-01 to 2025-12, **test** = 2026-01 to 2026-09

In [ ]:
import os
import sys
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from tqdm.auto import tqdm

sys.path.append("../src")
from preprocess import (COLUMNS, PICKUP, DROPOFF, RATE_CODES, TEST_START, TEST_END,
                        MIN_DURATION, MAX_DURATION, MIN_DISTANCE, MAX_DISTANCE, MIN_FARE, MAX_FARE,
                        clean, extract_features, month_of)

FIG_DIR = "../results/eda"
os.makedirs(FIG_DIR, exist_ok=True)

# same months as preprocess.py
files = [f for f in sorted(glob.glob("../data/yellow_tripdata_*.parquet")) if month_of(f) <= TEST_END]

# refine plots
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.titleweight": "bold"})
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
YEAR_COLOURS = {"2024": "tab:blue", "2025": "tab:green", "2026": "tab:orange"}
RATE_NAMES = {0: "missing", 1: "standard", 2: "JFK", 3: "Newark", 4: "Nassau/Westchester",
              5: "negotiated", 6: "group", 99: "unknown"}

## Monthly summary

`month_stats()` summaries monthly data.\
Results are cached in `results/eda/monthly_stats.pkl`.

In [ ]:
BIN = 0.5                                      
EDGES = np.arange(0, MAX_DURATION + BIN, BIN)    
NB = len(EDGES) - 1                              
# columns for the correlation matrix
NUMERIC = ["trip_distance", "fare_amount", "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
           "month_sin", "month_cos", "is_weekend", "duration_min"]


def sums(index, size, y):
    return np.stack([np.bincount(index, minlength=size),
                     np.bincount(index, y, minlength=size),
                     np.bincount(index, y * y, minlength=size)])


def month_stats(f):
    month = month_of(f)

    table = pq.read_table(f)
    nulls = {c: table.column(c).null_count for c in table.column_names} # nulls per column
    n_raw = table.num_rows # row count before cleaning
    raw = table.select(COLUMNS).to_pandas()
    del table

    # row dropping criteria
    duration = (raw[DROPOFF] - raw[PICKUP]).dt.total_seconds() / 60
    start = pd.Timestamp(f"{month}-01")
    problems = {
        "pickup outside month": ~raw[PICKUP].between(start, start + pd.DateOffset(months=1), inclusive="left"),
        "duration < 1 min": duration < MIN_DURATION,
        "duration > 3 hours": duration > MAX_DURATION,
        "distance < 0.1 mi": raw.trip_distance < MIN_DISTANCE,
        "distance > 60 mi": raw.trip_distance > MAX_DISTANCE,
        "fare < $3": raw.fare_amount < MIN_FARE,
        "fare > $250": raw.fare_amount > MAX_FARE,
        "RatecodeID not a known code": ~raw.RatecodeID.fillna(0).isin(RATE_CODES), # missing counts as code 0
    }
    problems = {name: int(rows.sum()) for name, rows in problems.items()}
    extremes = {"max distance": raw.trip_distance.max(), "min fare": raw.fare_amount.min(),
                "max fare": raw.fare_amount.max(), "min duration": duration.min(), "max duration": duration.max()}

    #cleaned trips
    d = clean(raw, f)
    del raw, duration
    y = d.duration_min.to_numpy()

    # used to index the histograms and sums
    hour = d[PICKUP].dt.hour.to_numpy() 
    day = d[PICKUP].dt.day_of_week.to_numpy()                                
    rate = np.searchsorted(RATE_CODES, d.RatecodeID.fillna(0).astype(int).to_numpy())  # position in RATE_CODES
    pu, do = d.PULocationID.to_numpy(), d.DOLocationID.to_numpy() # taxi zone IDs 
    b = np.minimum((y // BIN).astype(int), NB - 1) # duration bin 

    # numeric features
    feats = extract_features(d)
    numeric = np.column_stack([d.trip_distance, d.fare_amount] +
                              [feats[c].astype(float) for c in NUMERIC[2:-1]] + [y])

    return {
        "month": month,
        "columns": list(nulls),
        "nulls": nulls,
        "rows_raw": n_raw,
        "rows_clean": len(d),
        "problems": problems,
        "extremes": extremes,
        "hist_day_hour": np.bincount((day * 24 + hour) * NB + b, minlength=7 * 24 * NB).reshape(7, 24, NB),  
        "hist_rate": np.bincount(rate * NB + b, minlength=len(RATE_CODES) * NB).reshape(-1, NB),         
        "by_hour": sums(hour, 24, y),
        "by_rate": sums(rate, len(RATE_CODES), y),
        "by_pickup": sums(pu, 266, y),
        "by_route": sums(pu * 266 + do, 266 * 266, y),
        "corr": (len(numeric), numeric.sum(0), numeric.T @ numeric),
        # small sample of trips for the scatter plot
        "sample": d[["trip_distance", "RatecodeID", "duration_min"]].sample(min(3000, len(d)), random_state=0),
    }

In [ ]:
# load monthly summaries
CACHE = f"{FIG_DIR}/monthly_stats.pkl"
months = [month_of(f) for f in files]

# use cache or rebuild
stats = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else []
if [s["month"] for s in stats] != months:  # missing or out of date
    stats = [month_stats(f) for f in tqdm(files, unit=" month")]
    pickle.dump(stats, open(CACHE, "wb"))

# boolean masks over months
is_test = np.array([m >= TEST_START for m in months])
SPLITS = {"all": np.ones(len(stats), bool), "train": ~is_test, "test": is_test}


# for combining months 
def total(key, split="all"):
    return sum(s[key] for s, keep in zip(stats, SPLITS[split]) if keep)


def hist_quantile(h, q=0.5):
    c = np.cumsum(h)
    target = q * c[-1]                
    i = np.searchsorted(c, target)     
    below = c[i - 1] if i else 0  
    return EDGES[i] + BIN * (target - below) / h[i]


def r2(groups, split="train"):
    n, s, ss = total(groups, split)
    within = (ss - np.divide(s ** 2, n, out=np.zeros_like(s), where=n > 0)).sum()
    return 1 - within / (ss.sum() - s.sum() ** 2 / n.sum())


# one row per month, for the plots
monthly = pd.DataFrame({
    "rows_raw": [s["rows_raw"] for s in stats],
    "rows_clean": [s["rows_clean"] for s in stats],
    "mean_duration": [s["by_hour"][1].sum() / s["by_hour"][0].sum() for s in stats],  # total minutes / trips
    "median_duration": [hist_quantile(s["hist_rate"].sum(0)) for s in stats],
}, index=months)
monthly["kept"] = monthly.rows_clean / monthly.rows_raw  # share of raw trips that survive cleaning
print(f"{len(stats)} months: {len(stats) - is_test.sum()} train, {is_test.sum()} test")

## 1. Raw data analysis

In [ ]:
print(f"{monthly.rows_raw.sum():,} trips")
for split in ("train", "test"):
    print(f"  {split}: {monthly.rows_raw[SPLITS[split]].sum():,}")

monthly.rows_raw.div(1e6).plot.bar(figsize=(12, 3), color=[YEAR_COLOURS[m[:4]] for m in months],
                                   title="Trips per month (millions): blue 2024, green 2025, orange 2026 (test)")
plt.savefig(f"{FIG_DIR}/01_monthly_volume.png", bbox_inches="tight")

In [ ]:
has = pd.DataFrame({s["month"]: dict.fromkeys(s["columns"], True) for s in stats}).T.fillna(False)
for col in has.columns[~has.all()]:
    print(f"{col!r} only in {has.index[has[col]][0]} to {has.index[has[col]][-1]}")
print("columns the pipeline uses present in every month:", has[COLUMNS].all().all())

In [ ]:
# % missing per column, by year
nulls = pd.DataFrame([s["nulls"] for s in stats], index=months).fillna(0)
by_year = nulls.groupby(nulls.index.str[:4]).sum().div(monthly.rows_raw.groupby(monthly.index.str[:4]).sum(), axis=0) * 100
by_year.loc[:, by_year.sum() > 0].round(2)

In [ ]:
nulls.RatecodeID.div(monthly.rows_raw).mul(100).plot.bar(
    figsize=(12, 3), color=[YEAR_COLOURS[m[:4]] for m in months], title="% of trips missing RatecodeID")
plt.savefig(f"{FIG_DIR}/01_missing_ratecode.png", bbox_inches="tight")

In [ ]:
# the most extreme raw values, by year
extremes = pd.DataFrame([s["extremes"] for s in stats], index=months)
extremes.groupby(extremes.index.str[:4]).agg({"max distance": "max", "min fare": "min", "max fare": "max",
                                              "min duration": "min", "max duration": "max"}).round(1)

**Notes:** 
- 123.5M trips over 33 months (89.9M train, 33.6M test). 
- Files aren't all the same shape: `cbd_congestion_fee` appears from 2025-01 and `request_source` from 2026-06.
- Every column the pipeline reads is in every month.

**Missing values**. 
- `RatecodeID`, `passenger_count` and other sparse columns are missing on the same rows
-  Share increases from ~10% in 2024 to ~27% in 2026.
- Broken values turn up every year: 400,000-mile trips, fares up to \$863k, and durations of -300,000 minutes.

## 2. Bad rows

In [ ]:
# % of raw rows hitting each rule
problems = pd.DataFrame([s["problems"] for s in stats], index=months)
rates = pd.DataFrame({split: problems[SPLITS[split]].sum() / monthly.rows_raw[SPLITS[split]].sum()
                      for split in ("train", "test")})
print(rates.map("{:.2%}".format))
print()
for split in ("all", "train", "test"):
    m = monthly[SPLITS[split]]
    print(f"{split:5} kept {m.rows_clean.sum():,} of {m.rows_raw.sum():,} trips ({m.rows_clean.sum() / m.rows_raw.sum():.1%})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 3.5))
monthly.kept.mul(100).plot(ax=axes[0], marker="o", title="% of trips kept after cleaning")
axes[0].axvline(months.index(TEST_START), color="grey", ls="--")
axes[0].set_xticks(range(0, len(months), 6), months[::6])

problems.div(monthly.rows_raw, axis=0).mul(100).plot(ax=axes[1], title="% of trips hitting each rule")
axes[1].axvline(months.index(TEST_START), color="grey", ls="--")
axes[1].set_xticks(range(0, len(months), 6), months[::6])
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_cleaning.png", bbox_inches="tight")

**Notes:** 
- Cleaning keeps **92.8%** of trips (92.3% train, 94.1% test) 
- Moves a lot from month to month: about 95-96% (2024), to 84% in Nov 2025, then back to 93% in Dec 2025.

- Almost all of that comes from the `fare < $3` rule. This drops 1-2% of trips in 2024, 4-12% in 2025, then under 1% from Dec 2025 onwards. 
- Zero-distance and sub-1-minute trips roughly double between 2024 and 2026.

## 3. Trip duration (the target)

In [ ]:
target = {}
for split in ("train", "test"):
    h = total("hist_rate", split).sum(0)
    n, s, ss = total("by_hour", split).sum(1)
    target[split] = {"trips": n, "mean": s / n, "std": np.sqrt(ss / n - (s / n) ** 2),
                     **{f"p{q}": hist_quantile(h, q / 100) for q in (5, 25, 50, 75, 95, 99)}}
    plt.stairs(h / h.sum() / BIN, EDGES, label=f"{split} (median {target[split]['p50']:.1f} min)", lw=1.5)

plt.gcf().set_size_inches(9, 3.5)
plt.xlim(0, 90)
plt.legend()
plt.xlabel("minutes (cut off at 90)"); plt.ylabel("share of trips per minute")
plt.title("Trip duration")
plt.savefig(f"{FIG_DIR}/03_target.png", bbox_inches="tight")
pd.DataFrame(target).round(2)

**Notes:** 
- Most trips are short (train median 13.4 min, mean 17.1). 
- Test trips are about a minute longer through the middle (median 14.3).
- The tail will push RMSE up, so MAE is worth looking at too.

## 4. Time of day and day of week

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))

for split, ls in (("train", "-"), ("test", "--")):
    h = total("hist_day_hour", split)
    for name, days in (("weekdays", slice(0, 5)), ("weekends", slice(5, 7))):
        by_hour = [hist_quantile(h[days, hour].sum(0)) for hour in range(24)]
        axes[0].plot(by_hour, ls=ls, marker="o", ms=3, label=f"{name} ({split})",
                     color="tab:blue" if name == "weekdays" else "tab:orange")
axes[0].set(title="Median duration by hour", xlabel="pickup hour", ylabel="minutes")
axes[0].legend(fontsize=8)

h = total("hist_day_hour", "train")
by_day = np.array([hist_quantile(h[day].sum(0)) for day in range(7)])
axes[1].bar(DAYS, by_day, color=["tab:blue"] * 5 + ["tab:orange"] * 2)
axes[1].set(title="Median duration by day (train)", ylabel="minutes", ylim=(by_day.min() - 1, by_day.max() + 0.5))

counts = h.sum((0, 2)) / len(np.flatnonzero(SPLITS["train"])) / 1000
axes[2].fill_between(range(24), counts, alpha=0.4)
axes[2].plot(counts)
axes[2].set(title="Trips by pickup hour (train, per month)", xlabel="pickup hour", ylabel="thousand trips")

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_time.png", bbox_inches="tight")

In [ ]:
heat = np.apply_along_axis(hist_quantile, 2, total("hist_day_hour", "train"))
plt.figure(figsize=(12, 3))
plt.imshow(heat, aspect="auto", cmap="magma")
plt.colorbar(label="minutes")
plt.yticks(range(7), DAYS)
plt.xticks(range(24))
plt.grid(False)
plt.xlabel("pickup hour")
plt.title("Median duration by weekday and hour (train)")
plt.savefig(f"{FIG_DIR}/04_heatmap.png", bbox_inches="tight")

**Notes:** 
- Weekday trips are slowest from about 11am to 3pm (about 15 min). 
- Weekend mornings are quickest (about 10.4 min). 
- Trips spike at 4-5am
- Thursday is the slowest day and Sunday the quickest, with a gap of ~1 minute. 
- The busiest hour is 6pm.
- Test lines (dashed) have the same shape but are about a minute higher at almost every hour.

## 5. Trip destinations

In [ ]:
rate_table = {}
for split in ("train", "test"):
    h = total("hist_rate", split)
    rate_table[f"share {split}"] = h.sum(1) / h.sum()
    rate_table[f"median {split}"] = [hist_quantile(row) if row.sum() else np.nan for row in h]
rate_table = pd.DataFrame(rate_table, index=[RATE_NAMES[c] for c in RATE_CODES]).sort_values("median train")

rate_table["median train"].plot.barh(figsize=(7, 3.5), color="tab:purple", title="Median trip duration by rate code (train)")
plt.xlabel("minutes"); plt.ylabel("")
plt.savefig(f"{FIG_DIR}/05_rate_codes.png", bbox_inches="tight")
rate_table.style.format({"share train": "{:.2%}", "share test": "{:.2%}", "median train": "{:.1f}", "median test": "{:.1f}"})

In [ ]:
# rate code mix over time
mix = pd.DataFrame([s["hist_rate"].sum(1) / s["rows_clean"] for s in stats], index=months,
                   columns=[RATE_NAMES[c] for c in RATE_CODES])
ax = mix[["missing", "JFK", "Newark", "unknown", "negotiated"]].mul(100).plot(figsize=(12, 3.5), marker="o", ms=3,
                                                                             title="% of cleaned trips by rate code")
ax.axvline(months.index(TEST_START), color="grey", ls="--")
ax.set_xticks(range(0, len(months), 6), months[::6])
plt.savefig(f"{FIG_DIR}/05_rate_mix.png", bbox_inches="tight")

In [ ]:
# most popular routes, all months
n, s, _ = total("by_route")
top = np.argsort(n)[::-1][:10]
pd.DataFrame({"PULocationID": top // 266, "DOLocationID": top % 266, "trips": n[top],
              "share": n[top] / n.sum(), "mean duration": s[top] / n[top]}).round(3)

In [ ]:
# how much of the variation in duration can be explained by guessing groups average
def r2_out_of_sample(groups):
    n, s, _ = total(groups, "train")
    mean = np.where(n > 0, s / np.maximum(n, 1), s.sum() / n.sum())
    n_t, s_t, ss_t = total(groups, "test")
    sse = (ss_t - 2 * mean * s_t + n_t * mean ** 2).sum()
    return 1 - sse / (ss_t.sum() - s_t.sum() ** 2 / n_t.sum())

groupings = {"hour only": "by_hour", "rate code only": "by_rate", "pickup zone": "by_pickup", "route (PU, DO)": "by_route"}
pd.DataFrame({"train": {k: r2(g) for k, g in groupings.items()},
              "train -> test": {k: r2_out_of_sample(g) for k, g in groupings.items()}}).round(3)

**Notes:** 
- Trips with a missing rate code go from 14.5% of train to 25.1% of test, and standard trips fall from 80% to 69%. 
- Missing-code trips run longer than standard ones (median 16-17 vs 12 min)
- The `unknown` (99) code almost disappears in 2026-06, the same month `request_source` appears. 
- JFK trips take about 51 min against 12 for standard ones.

- Route alone explains 71% of the variation on train, and 69% when train averages are used to predict 2026.
- Rate code drops more (0.33 to 0.27)
- Hour alone explains under 2%.

## 6. Distance

In [ ]:
# 3,000 random trips from each training month
sample = pd.concat(s["sample"] for s, keep in zip(stats, SPLITS["train"]) if keep)
sample["rate"] = sample.RatecodeID.fillna(0).map(RATE_NAMES)

# colour the airport trips so they stand out
plt.figure(figsize=(8, 5))
for name, colour in [("standard", "tab:blue"), ("JFK", "tab:red"), ("Newark", "tab:green")]:
    d = sample[sample.rate == name]
    plt.scatter(d.trip_distance, d.duration_min, s=2, alpha=0.3, color=colour, label=name)
plt.xlim(0, 40); plt.ylim(0, 120)
plt.xlabel("trip distance (miles)"); plt.ylabel("duration (minutes)")
plt.title(f"Distance vs duration ({len(sample):,} trips sampled from train)")
plt.legend(markerscale=6)
plt.savefig(f"{FIG_DIR}/06_distance.png", bbox_inches="tight")


def correlations(split):
    # exact correlation matrix from n, column sums and X^T X
    n, s, xtx = (sum(part) for part in zip(*(st["corr"] for st, keep in zip(stats, SPLITS[split]) if keep)))
    mean = s / n
    cov = xtx / n - np.outer(mean, mean)
    sd = np.sqrt(np.diag(cov))
    return pd.DataFrame(cov / np.outer(sd, sd), index=NUMERIC, columns=NUMERIC)

corr = {split: correlations(split) for split in ("train", "test")}
corr["train"].loc[["trip_distance", "fare_amount", "duration_min"], ["trip_distance", "fare_amount", "duration_min"]].round(3)

**Notes:** 
- Across all of train, distance and fare are strongly correlated with duration (0.79 and 0.83) (can't be used, as unknown until trip is over)

## 7. Changes over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for year, colour in YEAR_COLOURS.items():
    d = monthly[monthly.index.str.startswith(year)]
    label = year + (" (test)" if year >= TEST_START[:4] else "")
    x = d.index.str[5:].astype(int)
    axes[0].plot(x, d.mean_duration, marker="o", color=colour, label=label, lw=2)
    axes[1].plot(x, d.rows_clean / 1e6, marker="o", color=colour, label=label, lw=2)
axes[0].set(title="Mean trip duration", ylabel="minutes")
axes[1].set(title="Cleaned trips per month", ylabel="millions")
for ax in axes:
    ax.set_xticks(range(1, 13), MONTHS)
    ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/07_over_time.png", bbox_inches="tight")

n, s, _ = np.array([s["by_hour"].sum(1) for s in stats]).T
pd.DataFrame({"trips": n, "s": s}, index=months).groupby(lambda m: m[:4]).sum().eval("mean_duration = s / trips").drop(columns="s")

In [ ]:
jan_sep = monthly[monthly.index.str[5:] <= "09"]
jan_sep.groupby(jan_sep.index.str[:4])[["mean_duration", "median_duration", "kept"]].mean().round(3)

**Notes:** 
- Yearly average goes 17.0, 17.2, 17.8 min
- Comparing only Jan-Sep gives 16.6, 16.7, 17.8
- 2024 and 2025 are almost the same, increase comes in 2026. 
- January is always quickest, and May and Sep-Dec are slowest. 
- Models will likely guess low on early 2026 as train/test split is by time.

## 8. The actual model features

In [ ]:
# correlation of each numeric feature with duration, over every trip in each split
pd.DataFrame({split: c["duration_min"].drop(["duration_min", "trip_distance", "fare_amount"])
              for split, c in corr.items()}).round(3)

**Notes:** 
- Features barely correlate with duration. 
- Strongest is `hour_cos` at about -0.10, the rest are under 0.05. 
- Month features flip between train and test (`month_sin` -0.05 vs 0.00), partly because test only covers Jan-Sep.

## 9. Test vs training sets

Models learn from 2024-2025 and get scored on 2026

In [ ]:
def share(key, split):
    n = total(key, split)[0]
    return n / n.sum()

# total variation distance
shift = {name: 0.5 * np.abs(share(key, "train") - share(key, "test")).sum()
         for name, key in [("pickup hour", "by_hour"), ("rate code", "by_rate"),
                           ("pickup zone", "by_pickup"), ("route", "by_route")]}
print("how different is test from train (0 = identical, 1 = no overlap):")
for name, value in shift.items():
    print(f"  {name:12} {value:.3f}")

n_train, n_test = total("by_route", "train")[0], total("by_route", "test")[0]
print()
print(f"routes seen in train: {(n_train > 0).sum():,}, in test: {(n_test > 0).sum():,}")
for k in (1, 30):
    rare = n_train < k
    print(f"test trips on a route with < {k} train trips: {n_test[rare].sum() / n_test.sum():.3%}")

In [ ]:
# pickup zones that grew or shrank the most between train and test
zones = pd.DataFrame({"train": share("by_pickup", "train"), "test": share("by_pickup", "test")})
zones["change"] = zones.test - zones.train
moved = zones.loc[zones.change.abs().sort_values(ascending=False).index[:12]].sort_values("change")
moved.change.mul(100).plot.barh(figsize=(7, 4), color=np.where(moved.change > 0, "tab:orange", "tab:blue"),
                                title="Biggest changes in pickup zone share, test minus train")
plt.xlabel("percentage points"); plt.ylabel("PULocationID")
plt.savefig(f"{FIG_DIR}/09_zone_shift.png", bbox_inches="tight")
moved.mul(100).round(2)

**Notes:** 
- Only 0.009% of 2026 trips are on a route never seen in training
- 0.5% are on routes with fewer than 30 training trips
- The pickup hour and zone mixes barely move (0.03 and 0.05). 
- Biggest shift is rate code (0.12), which is the growth in missing `RatecodeID`.  
- Zone changes are small: JFK (132), LaGuardia (138) and Midtown (161, 162, 163, 230) each lose 0.3-0.5 points, and the Lower East Side and East Village (4, 79, 148) gain a little.

## Summary

- There are 123.5M trips. Cleaning keeps 92.8%, mostly because of the `fare < $3` rule.
- Missing `RatecodeID` grows from about 10% of trips in 2024 to about 27% in 2026. These trips run longer, and this is the biggest difference between train and test.
- The route alone explains about 70% of the variation, and still holds when 2024-25 averages are used on 2026. Almost no test trips are on routes the training data hasn't seen.
- Trips in 2026 are about a minute longer, mostly in Jan-May. The daily and weekly patterns keep the same shape.
- Duration has a long right tail (median 13.4 min, mean 17.1, p99 about 71 min), so RMSE will be dominated by a few long trips.
- Time features on their own are weak.
- Distance and fare correlate strongly with duration (0.79 and 0.83) but aren't known until the trip ends, so they're left out.

**Implications for modelling**
- Pickup and dropoff zone should be the core features.
- Tree models should do better than linear, because duration depends on combinations of features.
- Report MAE alongside RMSE because of the long tail.
- Models are expected to slightly under-predict on 2026, since 2026 trips are about a minute longer than 2024-25 ones on average.
- Treating missing `RatecodeID` as its own category (0) is important, because its share nearly doubles in test.
